# [9660] Homework 3 - KNN
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9660/main/data/cardiovascular_disease_adults_60K.csv

## Homework Submission Rules (for all homework assignments)
* Homework is due by 6:05 PM on the due date
  * No late submission will be accepted
* You must submit a cleanly executed notebook (*.ipynb)
  * Verify that you are submitting the correct homework file
* Homework file naming convention
  * LastName_FirstName_HwX.ipynb  [Replace X with the homework #]
    * 1 point deducted for submitting homework not complying with naming convention
* Before submission, execute "Kernel -> Restart Kernel and Run All Cells"
  * 1 point deducted for not submitting a cleanly executed notebook

## Homework 3 Requirements
* Load data
  * Do NOT use meaningless columns (e.g. 'id') as independent variables
* Identify missing values and use SimpleImputer to replace missing values
* Ordinal Encode independent variables: 'smoker', 'alcohol_drinker', 'physically_active', 'cholesterol' and 'glucose'
  * From a health perspective:
    * It is better to NOT BE a 'smoker', NOT BE an 'alcohol_drinker', and TO BE 'physically_active'
    * For 'cholesterol' and 'glucose', 'average' is better than 'above_average', which is better than 'high'
* Dummy (one-hot) independent variable: encode 'gender'
* Label encode dependent variable: 'cardiovascular_disease'
* Separate independent and dependent variables
* Standardize independent variables
* Split data into training and test sets
* Train KNeighborsClassifier (with default hyperparameters)
* Calculate accuracy for KNeighborsClassifier (with default hyperparameters)
* Re-train KNeighborsClassifier (change n_neighbors hyperparameter and at least one other hyperparameter)
  * NOTE: The objective of changing these hyperparameters is to improve model accuracy
    * If you used hyperparameter random_state in your initial model training, do NOT change this value during model retrainings
    * Do NOT re-split training and test sets during model retrainings
* Calculate accuracy for re-trained KNeighborsClassifier (with updated hyperparameters)

## Submitted by Yashasvi Bhati (emplid: 24559155)

In [1]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 10/28/24 00:47:05


### Import libraries

In [2]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

### Load data

Risk Factors for Cardiovascular Heart Disease

age: Age of participant (integer)  
gender : Gender of participant (string - male, female)  
height : Height measured in centimeters (integer)  
weight : Weight measured in kilograms (integer)  
systolic_bp  : Systolic blood pressure reading taken from patient (integer)  
diastolic_bp  : Diastolic blood pressure reading taken from patient (integer)  
cholesterol : Total cholesterol level (string - average, above-average, high)  
glucose : Glucose level (string - average, above-average, high)  
smoker : Whether person smokes or not (string - N, Y)  
alcohol_drinker : Whether person drinks alcohol or not (string - NO, YES)  
physically_active : Whether person is physically active or not (string - no, yes)  
cardiovascular_disease : Whether person suffers from cardiovascular diseases or not (string - No, Yes)

In [3]:
# Read cardiovascular_disease_adults_60K.csv into dataframe
#  NOTES:
#   Field separator is '|'
#   Use column 'id' as index_col


### Examine data

In [4]:
# setting the column 'id' as index so that it is not used in the mdoel training
disease = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9660/main/data/cardiovascular_disease_adults_60K.csv', sep='|', index_col='id')

In [5]:
# displaying the first few rows of the dataset
disease.head(5)

,age,gender,height,weight,systolic_bp,diastolic_bp,cholesterol,glucose,smoker,alcohol_drinker,physically_active,cardiovascular_disease
id,,,,,,,,,,,,
0,50.0,Male,168.0,62.0,110,80,average,average,N,NO,yes,No
1,55.0,Female,156.0,85.0,140,90,high,average,N,NO,yes,Yes
2,51.0,Female,165.0,64.0,130,70,high,average,N,NO,no,Yes
3,48.0,Male,169.0,82.0,150,100,average,average,N,NO,yes,Yes
4,47.0,Female,156.0,56.0,100,60,average,average,N,NO,no,No


### Prepare data for model training

#### Use the SimpleImputer to replace missing values

In [6]:
# Check for missing values
disease.isnull().sum()

,0
age,139
gender,167
height,229
weight,74
systolic_bp,0
diastolic_bp,0
cholesterol,195
glucose,0
smoker,84
alcohol_drinker,0


In [7]:
# checking the datatypes to understand column requirement better
disease.dtypes

,0
age,float64
gender,object
height,float64
weight,float64
systolic_bp,int64
diastolic_bp,int64
cholesterol,object
glucose,object
smoker,object
alcohol_drinker,object


In [8]:
# using simple imputer to replace the null values
# age, height, weight > mean
# gender, cholestrol, smoker > mode


imp_mean = SimpleImputer(missing_values=np.nan, strategy='mean')
cols_to_impute_1 = ['age','height','weight']
disease[cols_to_impute_1] = imp_mean.fit_transform(disease[cols_to_impute_1])


imp_most_freq = SimpleImputer(missing_values=np.nan, strategy='most_frequent')
cols_to_impute_2 = ['gender', 'cholesterol', 'smoker']
disease[cols_to_impute_2] = imp_most_freq.fit_transform(disease[cols_to_impute_2])

#### Check for missing values again

In [9]:
# missing values
disease.isnull().sum()

,0
age,0
gender,0
height,0
weight,0
systolic_bp,0
diastolic_bp,0
cholesterol,0
glucose,0
smoker,0
alcohol_drinker,0


#### Ordinal Encode 'smoker', 'alcohol_drinker', 'physically_active', 'cholesterol' and 'glucose'

In [10]:
# to ordinal encode, lets start by seeing unique values for each class
#disease['smoker'].unique()
#disease['cholesterol'].unique()
#disease['glucose'].unique()
#disease['alcohol_drinker'].unique()
#disease['physically_active'].unique()

In [11]:
smoker_order = ['N', 'Y']
alcohol_drinker_order = ['NO', 'YES']
physically_active_order = ['yes', 'no']
cholesterol_order = ['average', 'above_average', 'high']
glucose_order = ['average', 'above_average', 'high']

encoder = OrdinalEncoder(categories=[smoker_order, alcohol_drinker_order, physically_active_order, cholesterol_order, glucose_order])

disease[['smoker', 'alcohol_drinker', 'physically_active', 'cholesterol', 'glucose']] = encoder.fit_transform(
    disease[['smoker', 'alcohol_drinker', 'physically_active', 'cholesterol', 'glucose']]
).astype(int)


In [12]:
# Display first few rows of updated dataframe
disease.head(5)

,age,gender,height,weight,systolic_bp,diastolic_bp,cholesterol,glucose,smoker,alcohol_drinker,physically_active,cardiovascular_disease
id,,,,,,,,,,,,
0,50.0,Male,168.0,62.0,110,80,0,0,0,0,0,No
1,55.0,Female,156.0,85.0,140,90,2,0,0,0,0,Yes
2,51.0,Female,165.0,64.0,130,70,2,0,0,0,1,Yes
3,48.0,Male,169.0,82.0,150,100,0,0,0,0,0,Yes
4,47.0,Female,156.0,56.0,100,60,0,0,0,0,1,No


#### Dummy (one-hot) encode gender

In [13]:
# checking for unqique values
disease['gender'].unique()

# applying one-hot coding
disease = pd.get_dummies(disease, columns=['gender'], dtype=int)


In [14]:
# Display first few rows of updated dataframe
disease.head(5)

,age,height,weight,systolic_bp,diastolic_bp,cholesterol,glucose,smoker,alcohol_drinker,physically_active,cardiovascular_disease,gender_Female,gender_Male
id,,,,,,,,,,,,,
0,50.0,168.0,62.0,110,80,0,0,0,0,0,No,0,1
1,55.0,156.0,85.0,140,90,2,0,0,0,0,Yes,1,0
2,51.0,165.0,64.0,130,70,2,0,0,0,1,Yes,1,0
3,48.0,169.0,82.0,150,100,0,0,0,0,0,Yes,0,1
4,47.0,156.0,56.0,100,60,0,0,0,0,1,No,1,0


#### Label encode target variable 'cardiovascular_disease'

In [15]:
le = LabelEncoder()
disease['cardiovascular_disease'] = le.fit_transform(disease['cardiovascular_disease'])

In [16]:
# Display first few rows of updated dataframe
disease.head(5)

,age,height,weight,systolic_bp,diastolic_bp,cholesterol,glucose,smoker,alcohol_drinker,physically_active,cardiovascular_disease,gender_Female,gender_Male
id,,,,,,,,,,,,,
0,50.0,168.0,62.0,110,80,0,0,0,0,0,0,0,1
1,55.0,156.0,85.0,140,90,2,0,0,0,0,1,1,0
2,51.0,165.0,64.0,130,70,2,0,0,0,1,1,1,0
3,48.0,169.0,82.0,150,100,0,0,0,0,0,1,0,1
4,47.0,156.0,56.0,100,60,0,0,0,0,1,0,1,0


### Separate independent and dependent variables
* Independent variables: All remaining variables except 'cardiovascular_disease'
* Dependent variable: 'cardiovascular_disease'

In [17]:
X = disease.drop('cardiovascular_disease', axis=1) # Independent variables
y = disease['cardiovascular_disease']  # dependent / target variable

### Standardize independent variables

In [18]:
scaler = StandardScaler() # note : only independent variables are scaled
X = scaler.fit_transform(X)

### Split data into training and test sets

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, test_size = 0.3, random_state=42)

### Train KNeighborsClassifier (with default hyperparameters)


In [20]:
knn = KNeighborsClassifier()
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [21]:
knn.fit(X_train, y_train)

KNeighborsClassifier()

### Evaluate performance for KNeighborsClassifier (with default hyperparameters)

In [22]:
y_pred = knn.predict(X_test)

In [23]:
# Print model accuracy score
accuracy_score1 = accuracy_score(y_test, y_pred)   # knn.score is an alternative method for accuracy_score. If using accuracy_score use (y_test, y_pred) as arguments
print(f'Initial Accuracy = {round((accuracy_score1 * 100), 3)}%')

Initial Accuracy = 64.039%


### Train KNeighborsClassifier (change n_neighbors hyperparameter and at least one other hyperparameter)
NOTE: The objective of changing these hyperparameters is to improve model accuracy

In [24]:
knn2 = KNeighborsClassifier(n_neighbors=10, weights = 'distance', metric = 'manhattan')
knn2.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'manhattan',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 10,
 'p': 2,
 'weights': 'distance'}

In [25]:
knn2.fit(X_train, y_train)

KNeighborsClassifier(metric='manhattan', n_neighbors=10, weights='distance')

In [26]:
y_pred2 = knn2.predict(X_test)

### Evaluate performance for KNeighborsClassifier (with updated hyperparameters)

In [27]:
accuracy_score2 = accuracy_score(y_test, y_pred2)
print(f'Updated Accuracy = {round((accuracy_score2 * 100), 3)}%')

Updated Accuracy = 66.322%


In [28]:
# printing both accuracy before and after retaining the model with updated parameters
print(f'Initial Accuracy = {round((accuracy_score1 * 100), 3)}%')
print(f'Updated Accuracy = {round((accuracy_score2 * 100), 3)}%')

Initial Accuracy = 64.039%
Updated Accuracy = 66.322%
